<a href="https://colab.research.google.com/github/Priyanshi07-ai/mobile_app_review_classify_ann_model/blob/main/ann_final1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [175]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, GlobalAveragePooling1D, Dense, Dropout

In [176]:
df = pd.read_csv("UserFeedbackData.csv")

print(df.head())

  review_id                                            content  score  \
0       1_1  Ever since the update, there's a weird glitch ...      3   
1       1_2  Don't believe the news!!! You can absolutely c...      5   
2       1_3  Great app. Too many ads. If you saw a video an...      4   
3       1_4  Good app, but there's a glitch that I've had i...      3   
4       1_5  The creator of this app created an algorithm t...      5   

   TU_count                    app_id app_name  RC_ver  
0     22930  com.zhiliaoapp.musically   TikTok  29.6.4  
1     18518  com.zhiliaoapp.musically   TikTok  29.6.4  
2      3467  com.zhiliaoapp.musically   TikTok  29.8.4  
3      2157  com.zhiliaoapp.musically   TikTok  29.8.4  
4      1780  com.zhiliaoapp.musically   TikTok  29.8.4  


In [177]:
def label_sentiment(score):
    if score >= 4:
        return 2   # Positive
    elif score == 3:
        return 1   # Neutral
    else:
        return 0   # Negative

df['sentiment'] = df['score'].apply(label_sentiment)

print(df.head(15))

   review_id                                            content  score  \
0        1_1  Ever since the update, there's a weird glitch ...      3   
1        1_2  Don't believe the news!!! You can absolutely c...      5   
2        1_3  Great app. Too many ads. If you saw a video an...      4   
3        1_4  Good app, but there's a glitch that I've had i...      3   
4        1_5  The creator of this app created an algorithm t...      5   
5        1_6  App is ok but I've been having a consistent pr...      3   
6        1_7  Keeps freezing up my entire phone, I'll try to...      1   
7        1_8  Addictive but certainly not without its issues...      4   
8        1_9  The new updates have been extraordinarily bugg...      3   
9       1_10  I have encountered a rare glitch that no one s...      2   
10      1_11  Good toilet app. The algorithm has taught me s...      4   
11      1_12  The app is super glitchy. I also somehow got a...      1   
12      1_13  Tiktok provides a great 

In [178]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text)
    return text

df['content'] = df['content'].apply(preprocess)

In [179]:
import re

def normalize_text(text):
    text = text.lower()

    replacements = {
        "interface": "ui",
        "design": "ui",
        "layout": "ui",
        "look": "ui",
        "appearance": "ui",

        "experience": "ux",
        "confusing": "ux",
        "navigation": "ux",

        "crashing": "bug",
        "crashes": "bug",
        "crash": "bug",
        "error": "bug",
        "buggy": "bug",
        "glitch": "bug",

        "slow": "slow",
        "lag": "lag",
        "delay": "lag",

        "ads": "ads",
        "advertisement": "ads",
        "popup": "ads"
    }

    for word, replacement in replacements.items():
        text = re.sub(r'\b' + word + r'\b', replacement, text)

    return text

In [180]:
# for classification in the form of bug/ui/ux/performance/ads/slow/lag/crash

def label_topics(text):
    text = text.lower()
    negations = ["no", "not", "never", "without"]

    def has_word(words):
        for w in words:
            if re.search(rf"\b{w}\b", text):
                for neg in negations:
                    if re.search(rf"{neg}\s+{w}", text):
                        return 0
                return 1
        return 0

    bugs = has_word(["bug","crash","error","issue","lag","freeze"])
    ui = has_word([
      "ui", "design", "interface", "layout",
      "look", "appearance", "visual", "screen"
    ])
    ux = has_word(["easy","smooth","difficult","confusing"])
    ads = has_word(["ad","ads","popup","advertisement"])

    return [bugs, ui, ux, ads]

df[['bugs','ui','ux','ads']] = df['content'].apply(lambda x: pd.Series(label_topics(x)))

df['contents'] = df['content'].apply(normalize_text)
df['topic_multi'] = df['contents'].apply(label_topics)

In [181]:
neutral_words = ["okay", "average", "fine", "nothing special", "normal"]

def adjust_sentiment(text, label):
    for w in neutral_words:
        if w in text:
            return 1   # neutral
    return label

df['sentiment'] = df.apply(lambda x: adjust_sentiment(x['content'], x['sentiment']), axis=1)

In [182]:
y_sent = df['sentiment']   # already encoded (0,1,2)
y_topic = df[['bugs','ui','ux','ads']].values

In [183]:
# def handle_negation(text):
#     words = text.split()
#     result = []
#     negate = False

#     for w in words:
#         if w in ["not", "no", "never"]:
#             negate = True
#             result.append(w)
#         elif negate:
#             result.append("not_" + w)
#             negate = False
#         else:
#             result.append(w)

#     return " ".join(result)

# df['content'] = df['content'].apply(handle_negation)

In [184]:
tokenizer = Tokenizer(num_words=10000, oov_token="<OOV>")
tokenizer.fit_on_texts(df['content'])

sequences = tokenizer.texts_to_sequences(df['content'])
X = pad_sequences(sequences, maxlen=150)

In [185]:
# df_neg = df[df.sentiment == 0]
# df_neu = df[df.sentiment == 1]
# df_pos = df[df.sentiment == 2]

# max_size = max(len(df_neg), len(df_neu), len(df_pos))

# from sklearn.utils import resample

# df_neg_up = resample(df_neg, replace=True, n_samples=max_size, random_state=42)
# df_neu_up = resample(df_neu, replace=True, n_samples=max_size, random_state=42)
# df_pos_up = resample(df_pos, replace=True, n_samples=max_size, random_state=42)

# df = pd.concat([df_neg_up, df_neu_up, df_pos_up])
# df = df.sample(frac=1, random_state=42)
# print(df['sentiment'].value_counts())

In [186]:
# import numpy as np

# y_topic = np.array(df['topic_multi'].tolist())

# bug_count = np.sum(y_topic[:, 0])   # column 0 = bug
# print("Number of Bug reviews:", bug_count)
# ui_count = np.sum(y_topic[:, 1])   # column 0 = bug
# print("Number of Bug reviews:", ui_count)
# ux_count = np.sum(y_topic[:, 2])   # column 0 = bug
# print("Number of Bug reviews:", ux_count)
# perform_count = np.sum(y_topic[:, 3])   # column 0 = bug
# print("Number of Bug reviews:", perform_count)
# ads_count = np.sum(y_topic[:, 4])   # column 0 = bug
# print("Number of Bug reviews:", ads_count)

In [187]:
import numpy as np
y_topic = np.array(df['topic_multi'].tolist())
print(y_topic)

[[1 0 0 0]
 [0 0 0 0]
 [0 1 0 1]
 ...
 [0 0 0 0]
 [0 1 0 0]
 [0 0 0 0]]


In [188]:
# X = df['content']

# vectorizer = TfidfVectorizer(
#     max_features=10000,
#     ngram_range=(1,2),
#     stop_words='english',
#     min_df=2,
#     max_df=0.9
# )
# X = vectorizer.fit_transform(X).toarray()
# y = df['sentiment']

In [189]:
X_train, X_test, y_sent_train, y_sent_test, y_topic_train, y_topic_test = train_test_split(
    X, y_sent, y_topic, test_size=0.2, random_state=42
)

print("X_train:", X_train.shape)
print("y_train:", y_sent_train.shape)
print("y_train:", y_topic_train.shape)

X_train: (12000, 150)
y_train: (12000,)
y_train: (12000, 4)


In [190]:
input_layer = Input(shape=(150,))

x = Embedding(input_dim=10000, output_dim=128)(input_layer)
x = GlobalAveragePooling1D()(x)

x = Dense(128, activation='relu')(x)
x = Dropout(0.3)(x)

x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)

sentiment_output = Dense(3, activation='softmax', name='sentiment')(x)
topic_output = Dense(4, activation='sigmoid', name='topic')(x)

model = Model(inputs=input_layer, outputs=[sentiment_output, topic_output])

In [191]:
model.compile(
    optimizer='adam',
    loss={
        'sentiment': 'sparse_categorical_crossentropy',
        'topic': 'binary_crossentropy'
    },
    loss_weights={
        'sentiment': 1.0,
        'topic': 1.5
    },
    metrics={
        'sentiment': 'accuracy',
        'topic': 'accuracy'
    }
)

In [192]:
model.fit(
    X_train,
    {'sentiment': y_sent_train, 'topic': y_topic_train},
    epochs=10,
    batch_size=32,
    validation_data=(X_test, {'sentiment': y_sent_test, 'topic': y_topic_test})
)

Epoch 1/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 9s 20ms/step - loss: 1.5852 - sentiment_accuracy: 0.5368 - sentiment_loss: 1.0009 - topic_accuracy: 0.6617 - topic_loss: 0.3895 - val_loss: 1.6129 - val_sentiment_accuracy: 0.5567 - val_sentiment_loss: 1.0573 - val_topic_accuracy: 0.7680 - val_topic_loss: 0.3701
Epoch 2/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 16ms/step - loss: 1.3469 - sentiment_accuracy: 0.6722 - sentiment_loss: 0.8079 - topic_accuracy: 0.7172 - topic_loss: 0.3593 - val_loss: 1.2461 - val_sentiment_accuracy: 0.7027 - val_sentiment_loss: 0.7379 - val_topic_accuracy: 0.7730 - val_topic_loss: 0.3388
Epoch 3/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - loss: 1.1573 - sentiment_accuracy: 0.7219 - sentiment_loss: 0.6900 - topic_accuracy: 0.7184 - topic_loss: 0.3116 - val_loss: 1.1298 - val_sentiment_accuracy: 0.7213 - val_sentiment_loss: 0.6877 - val_topic_accuracy: 0.7857 - val_topic_loss: 0.2946
Epoch 4/10
375/375 ━━━━━━━━━━━━━━━━━━━━ 6s 17ms/step - loss: 0.9984 - sentiment_accuracy: 0

In [193]:
pred_sent, pred_topic = model.predict(X_test)

sentiment = pred_sent.argmax(axis=1)
topics = (pred_topic > 0.4).astype(int)

94/94 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [194]:
import re
from textblob import TextBlob

def clean_input(text):
    text = text.lower()

    # remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # spelling correction
    text = str(TextBlob(text).correct())

    return text


In [195]:
# import numpy as np

# def predict_review(text):
#     text = preprocess(text)

#     # text = label_topic_multi(text)
#     vec = tokenizer.transform([text]).toarray()

#     pred_sent, pred_topic = model.predict(vec)


#     # Sentiment (argmax)
#     sent_label = np.argmax(pred_sent)

#     sentiments = ["Negative", "Neutral", "Positive"]
#     sentiment_result = sentiments[sent_label]

#     # Topics (multi-label → threshold)
#     topic_pred = (pred_topic > 0.4).astype(int)[0]

#     topics = ["Bug", "UI", "Performance", "Ads", "UX"]

#     topic_result = [topics[i] for i in range(len(topics)) if topic_pred[i] == 1]

#     return sentiment_result, topic_result

In [196]:
def predict_review(text, model, tokenizer, maxlen=150):
    import numpy as np
    import re

    # 🔹 same preprocessing (VERY IMPORTANT)
    def preprocess(text):
        text = text.lower()
        text = re.sub(r'\d+', '', text)
        text = re.sub(r'\s+', ' ', text)
        return text

    text = preprocess(text)

    # 🔹 convert to sequence
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=maxlen)

    # 🔹 prediction
    pred_sent, pred_topic = model.predict(padded)

    # 🔹 sentiment
    sent_index = np.argmax(pred_sent, axis=1)[0]

    sentiment_map = {
        0: "Negative",
        1: "Neutral",
        2: "Positive"
    }

    sentiment = sentiment_map[sent_index]

    # 🔹 topics
    topic_labels = ["bugs", "ui", "ux", "ads"]
    topic_pred = (pred_topic > 0.3).astype(int)[0]

    topics = [topic_labels[i] for i in range(len(topic_labels)) if topic_pred[i] == 1]

    return sentiment, topics

In [201]:

text = "App crashes frequently and too many ads"
text1 = "App crashes every time I open it and too many ads"
text2 = "Very smooth and easy to use, great experience"
text3 = "The interface looks okay but nothing special"

sentiment, topics = predict_review(text, model, tokenizer)
sentiment1, topics1 = predict_review(text1, model, tokenizer)
sentiment2, topics2 = predict_review(text2, model, tokenizer)
sentiment3, topics3 = predict_review(text3, model, tokenizer)

print("Sentiment:", sentiment, "Topics:", topics)
print("Sentiment:", sentiment1, "Topics:", topics1)
print("Sentiment:", sentiment2, "Topics:", topics2)
print("Sentiment:", sentiment3, "Topics:", topics3)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Sentiment: Negative Topics: ['bugs', 'ads']
Sentiment: Negative Topics: ['bugs', 'ads']
Sentiment: Positive Topics: ['ux']
Sentiment: Neutral Topics: ['ui']
